## 1. Imports

In [ ]:
import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.cluster import DBSCAN

random.seed(42); np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.cmap'] = 'gray'

## 2. Configuració

Paràmetres validats empíricament sobre el dataset.

In [ ]:
RAW_DIR    = Path('data/raw')
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
N_SAMPLES  = 10

# Corner detector: 'harris' | 'shi_tomasi' | 'fast'
CORNER_METHOD   = 'harris'

# Harris
HARRIS_K        = 0.04
HARRIS_KSIZE    = 3
HARRIS_THRESH   = 0.005   # fraction of max response — lower = more corners

# Shi-Tomasi
ST_MAX_CORNERS  = 600
ST_QUALITY      = 0.005
ST_MIN_DIST     = 3

# FAST
FAST_THRESHOLD  = 15

# NMS: suppress corners within this radius (px)
NMS_RADIUS      = 5

# DBSCAN: key params — tune these if recall is low
DBSCAN_EPS      = 20    # max distance between corners in same cluster
DBSCAN_MIN_PTS  = 4     # min corners to form a cluster

# Shape filter
AREA_RATIO_MIN  = 0.0005
AREA_RATIO_MAX  = 0.10
ASPECT_MIN      = 1.5
ASPECT_MAX      = 9.0
MIN_WIDTH       = 25
MIN_HEIGHT      = 8

## 3. Preprocessament

In [ ]:
def preprocess(img_bgr):
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.0)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(blurred)

## 4. Detecció de corners

**Harris**: $R = \det(M) - k \cdot \text{tr}(M)^2$. Alta resposta on hi ha cantonada real en dues direccions.

**Shi-Tomasi**: $R = \min(\lambda_1, \lambda_2)$. Més estable que Harris.

**FAST**: Mira un cercle de 16px. Si N consecutius son tots mes brillants o foscos que el centre → corner.

In [ ]:
def detect_corners(gray):
    if CORNER_METHOD == 'harris':
        r       = cv2.cornerHarris(np.float32(gray), blockSize=2,
                                   ksize=HARRIS_KSIZE, k=HARRIS_K)
        ys, xs  = np.where(r > HARRIS_THRESH * r.max())
        pts     = np.column_stack([xs, ys]).astype(np.float32)
        scores  = r[ys, xs]

    elif CORNER_METHOD == 'shi_tomasi':
        p = cv2.goodFeaturesToTrack(gray, ST_MAX_CORNERS, ST_QUALITY, ST_MIN_DIST)
        pts    = p.reshape(-1, 2) if p is not None else np.empty((0, 2), np.float32)
        scores = np.ones(len(pts))

    elif CORNER_METHOD == 'fast':
        fast   = cv2.FastFeatureDetector_create(threshold=FAST_THRESHOLD)
        kps    = fast.detect(gray, None)
        pts    = np.array([[k.pt[0], k.pt[1]] for k in kps], np.float32) if kps else np.empty((0,2), np.float32)
        scores = np.array([k.response for k in kps]) if kps else np.empty(0)

    return pts, scores

## 5. NMS local sobre corners

Quan dos corners estan a menys de `NMS_RADIUS` píxels, quedem-nos amb el de **major resposta** i eliminem els veïns. Redueix redundàncies i fa que DBSCAN treballi millor.

In [ ]:
def nms_corners(pts, scores, radius=NMS_RADIUS):
    if len(pts) == 0:
        return pts, scores
    order          = np.argsort(scores)[::-1]
    pts, scores    = pts[order], scores[order]
    keep           = np.ones(len(pts), dtype=bool)
    for i in range(len(pts)):
        if not keep[i]: continue
        dists           = np.linalg.norm(pts[i+1:] - pts[i], axis=1)
        keep[i+1:][dists < radius] = False
    return pts[keep], scores[keep]

## 6. Clustering DBSCAN → bounding boxes

DBSCAN agrupa corners pròxims. Cada cluster → bounding box dels seus punts extrems.

- `eps` petit → clusters petits i locals (millor per matrícules petites, perill de fragmentar)
- `eps` gran → clusters grans (pot fusionar matrícula amb entorn)

Outliers (label=-1) es descarten.

In [ ]:
def cluster_to_boxes(pts):
    if len(pts) < DBSCAN_MIN_PTS:
        return []
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_PTS).fit_predict(pts)
    boxes  = []
    for lbl in set(labels):
        if lbl == -1: continue
        cp = pts[labels == lbl]
        x1, y1 = int(cp[:, 0].min()), int(cp[:, 1].min())
        x2, y2 = int(cp[:, 0].max()), int(cp[:, 1].max())
        if x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2-x1, y2-y1))
    return boxes

## 7. Filtre per forma

In [ ]:
def filter_by_shape(boxes, img_shape):
    H, W     = img_shape[:2]
    img_area = H * W
    kept     = []
    for (x, y, w, h) in boxes:
        if w < MIN_WIDTH or h < MIN_HEIGHT: continue
        if not (AREA_RATIO_MIN <= w*h / img_area <= AREA_RATIO_MAX): continue
        if not (ASPECT_MIN <= w / float(h) <= ASPECT_MAX): continue
        kept.append((x, y, w, h))
    return kept

## 8. Pipeline completa

In [ ]:
def detect_plates_corners(img_bgr):
    enhanced        = preprocess(img_bgr)
    pts, scores     = detect_corners(enhanced)
    pts_nms, sc_nms = nms_corners(pts, scores)
    raw_boxes       = cluster_to_boxes(pts_nms)
    final_boxes     = filter_by_shape(raw_boxes, img_bgr.shape)
    return {
        'enhanced':  enhanced,
        'pts_raw':   pts,
        'pts_nms':   pts_nms,
        'raw_boxes': raw_boxes,
        'boxes':     final_boxes,
    }

## 9. Visualització (totes les fases)

In [ ]:
def show_pipeline_stages(img_bgr, result, title=''):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0,0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0,0].set_title('Original'); axes[0,0].axis('off')

    axes[0,1].imshow(result['enhanced'])
    axes[0,1].set_title('1. Preprocessat'); axes[0,1].axis('off')

    axes[0,2].imshow(result['enhanced'])
    if len(result['pts_raw']) > 0:
        axes[0,2].scatter(result['pts_raw'][:,0], result['pts_raw'][:,1],
                          s=2, c='yellow', alpha=0.5)
    axes[0,2].set_title(f"2. Corners raw ({len(result['pts_raw'])})"); axes[0,2].axis('off')

    axes[1,0].imshow(result['enhanced'])
    if len(result['pts_nms']) > 0:
        axes[1,0].scatter(result['pts_nms'][:,0], result['pts_nms'][:,1],
                          s=5, c='lime', alpha=0.8)
    axes[1,0].set_title(f"3. Corners NMS ({len(result['pts_nms'])})"); axes[1,0].axis('off')

    axes[1,1].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in result['raw_boxes']:
        axes[1,1].add_patch(patches.Rectangle((x,y), w, h, linewidth=1,
                            edgecolor='orange', facecolor='none'))
    axes[1,1].set_title(f"4. Clusters raw ({len(result['raw_boxes'])})"); axes[1,1].axis('off')

    axes[1,2].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in result['boxes']:
        axes[1,2].add_patch(patches.Rectangle((x,y), w, h, linewidth=2,
                            edgecolor='lime', facecolor='none'))
    axes[1,2].set_title(f"5. Final boxes ({len(result['boxes'])})"); axes[1,2].axis('off')

    plt.tight_layout(); plt.show()

## 10. Execució sobre 10 imatges aleatòries

In [ ]:
all_images     = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in VALID_EXTS])
print(f'Found {len(all_images)} images')
selected_paths = random.sample(all_images, min(N_SAMPLES, len(all_images)))

results = []
for path in selected_paths:
    img = cv2.imread(str(path))
    if img is None: continue
    res = detect_plates_corners(img)
    results.append((path, img, res))
    print(f"{path.name:30s} corners={len(res['pts_nms']):4d}  "
          f"clusters={len(res['raw_boxes']):3d}  final={len(res['boxes']):2d}")

In [ ]:
for path, img, res in results:
    show_pipeline_stages(img, res, title=path.name)

## 11. Vista resum

In [ ]:
n = len(results); cols = 2; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 5*rows))
axes = np.atleast_2d(axes).flatten()
for ax, (path, img, res) in zip(axes, results):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in res['boxes']:
        ax.add_patch(patches.Rectangle((x,y), w, h, linewidth=2,
                     edgecolor='lime', facecolor='none'))
    ax.set_title(f"{path.name} -- {len(res['boxes'])} boxes")
    ax.axis('off')
for ax in axes[len(results):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 12. Comparació dels 3 detectors sobre la mateixa imatge

Executa per veure visualment les diferències entre Harris, Shi-Tomasi i FAST.

In [ ]:
test_img = cv2.imread(str(selected_paths[0]))
methods  = ['harris', 'shi_tomasi', 'fast']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Comparació detectors — {selected_paths[0].name}', fontsize=14)
orig_method = CORNER_METHOD
for ax, method in zip(axes, methods):
    globals()['CORNER_METHOD'] = method
    enhanced = preprocess(test_img)
    pts, sc  = detect_corners(enhanced)
    pts_f, _ = nms_corners(pts, sc)
    ax.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    if len(pts_f) > 0:
        ax.scatter(pts_f[:,0], pts_f[:,1], s=4, c='lime', alpha=0.8)
    ax.set_title(f'{method} ({len(pts_f)} corners after NMS)')
    ax.axis('off')
globals()['CORNER_METHOD'] = orig_method
plt.tight_layout(); plt.show()

## 13. Notes per ajustar

**Poc recall (matrícules no detectades):**
- Baixa `HARRIS_THRESH` (ex: `0.003`) per detectar corners més dèbils.
- Augmenta `DBSCAN_EPS` (ex: `25`) per unir clusters propers.
- Baixa `DBSCAN_MIN_PTS` (ex: `3`) per acceptar clusters menys densos.
- Augmenta `DBSCAN_EPS` és especialment útil per matrícules amb pocs corners (petites o poc contrast).

**Massa falsos positius:**
- Augmenta `HARRIS_THRESH` (ex: `0.01`).
- Augmenta `DBSCAN_MIN_PTS` (ex: `6`).
- Estreu `ASPECT_MIN`/`ASPECT_MAX` (ex: `2.0`/`7.0`).

**Limitació coneguda**: matrícules molt petites (<20px d'alçada) generen pocs corners concentrats en una sola fila. DBSCAN els considera outliers i no forma cluster. Per aquests casos, la pipeline morfològica és preferible.